# Ноутбук 01 — Смотрим на данные глазами

**Цель:** понять, что лежит в `hard_train.csv` и `hard_test.csv`, не обучая ничего.

Словарик (запомни эти 5 слов):
- **Строка (row)** — один клиент.
- **Столбец/колонка (column)** — один признак или ответ.
- **DataFrame** — таблица pandas (то, во что превращается CSV после чтения).
- **NaN** — пропуск, пустая клетка (Not a Number).
- **dtype** — тип колонки: число (`int64`/`float64`) или текст (`object`).

In [ ]:
# Переходим в корень проекта (если ноутбук открыт из папки notebooks).
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Рабочая папка:", os.getcwd())
# Импортируем pandas (таблицы) и даём ей короткое имя pd — так принято везде.
import pandas as pd
# Читаем оба CSV: train (с ответами) и test (без ответов).
train = pd.read_csv("hard_train.csv")
test = pd.read_csv("hard_test.csv")
# Печатаем размеры: (строки, столбцы).
print("train:", train.shape, " test:", test.shape)

Ожидаем: train — (10000, 61), test — (2000, 60). У теста на одну колонку меньше, потому что в нём нет ответа `protection_score` — его мы и предсказываем.

## 1. Список всех колонок

In [ ]:
# .columns — все имена колонок. enumerate нумерует их с 1 для удобного чтения.
for number, name in enumerate(train.columns, start=1):
    print(f"{number:2d}. {name}")  # :2d — выровнять номер по ширине 2 символа.

Три особые колонки: `customer_id` (ID, не признак), `protection_score` (ответ, есть только в train), остальное — признаки. Текстовые признаки легко узнать: `gender`, `region`, `city_type`, `education`, `family_status`, `employment`, `wealth_segment` — всего 7 штук.

## 2. Типы колонок: где числа, где текст

In [ ]:
# .dtypes — тип каждой колонки. value_counts() считает, сколько колонок каждого типа.
print(train.dtypes.value_counts())
# Отберём только текстовые колонки: select_dtypes(include=object) — взять колонки типа object (строки).
text_columns = train.select_dtypes(include="object").columns.tolist()  # .tolist() — в обычный список Python.
print("Текстовые колонки:", text_columns)
# А это числовые колонки: всё, что НЕ текст и НЕ особые колонки.
numeric_columns = [c for c in train.columns if c not in text_columns + ["protection_score"]]
print("Числовых признаков:", len(numeric_columns))

## 3. Первые строки и случайные строки

In [ ]:
# .head(3) — первые 3 строки таблицы. Удобно, чтобы увидеть формат значений.
train.head(3)

In [ ]:
# .sample(3, random_state=0) — 3 СЛУЧАЙНЫЕ строки (random_state фиксирует, какие именно).
# Полезно: первые строки могут быть не похожи на остальные.
train.sample(3, random_state=0)

## 4. Описательные статистики: describe()

In [ ]:
# .describe() считает статистики по каждой ЧИСЛОВОЙ колонке. Разберём строки вывода:
# count — сколько непустых значений; mean — среднее; std — разброс (станд. отклонение);
# min/max — минимум/максимум; 25%/50%/75% — квартили (50% = медиана).
# .T в конце переворачивает таблицу (колонки становятся строками) — так читается легче.
train.describe().T.head(15)  # Покажем первые 15 признаков, чтобы вывод не был гигантским.

Обрати внимание: у `house_value` count гораздо меньше 10000 — там много пропусков. У флагов типа `mortgage` max = 1.0 — это колонки 0/1.

## 5. Наша цель: protection_score

In [ ]:
# Вытаскиваем колонку цели в отдельную переменную y (так принято: X — признаки, y — ответ).
y = train["protection_score"]
# Описательные статистики именно цели.
print(y.describe())
# Проверяем границы: все ли значения внутри [0, 100]?
# .between(0, 100) даёт True/False на каждую строку; .all() — True, если все True.
print("Все значения в [0, 100]:", y.between(0, 100).all())
# Есть ли пропуски в цели? .isna() — True где пропуск; .sum() — сколько их.
print("Пропусков в цели:", y.isna().sum())

## 6. Текстовые признаки: какие значения бывают

In [ ]:
# Для каждой текстовой колонки (кроме ID) покажем все уникальные значения и их частоты.
for column in ["gender", "region", "city_type", "education", "family_status", "employment", "wealth_segment"]:
    print(f"--- {column} ---")
    # .value_counts() считает, сколько раз встречается каждое значение (по убыванию частоты).
    print(train[column].value_counts())
    print()  # Пустая строка-разделитель для читаемости.

## 7. Пропуски: где и сколько

In [ ]:
# .isna() — True где пропуск; .mean() — доля пропусков в каждой колонке.
# Умножаем на 100, чтобы получить проценты; sort_values(ascending=False) — сначала худшие.
missing_train = train.isna().mean().mul(100).sort_values(ascending=False)
# Оставляем только колонки, где пропуски есть (> 0).
missing_train = missing_train[missing_train > 0]
print("Пропуски в train (%):")
print(missing_train.round(2))  # .round(2) — округлить до 2 знаков.
print()
# То же самое для теста — пропуски должны быть похожими (иначе тест странный).
missing_test = test.isna().mean().mul(100).sort_values(ascending=False)
missing_test = missing_test[missing_test > 0]
print("Пропуски в test (%):")
print(missing_test.round(2))

Пропуски только в трёх колонках: `house_value` (~47%), `car_value` (~41%), `average_claim_cost` (~40%). Это нормально: у кого нет дома/машины/обращений — тем ставить нечего. Как их заполнять — в ноутбуке 03.

## 8. Проверки целостности: ID, дубликаты, пересечения

In [ ]:
# 8а. ID должны быть уникальны: nunique() считает УНИКАЛЬНЫЕ значения, len — все строки.
print("Уникальных ID в train:", train["customer_id"].nunique(), "из", len(train))
print("Уникальных ID в test:", test["customer_id"].nunique(), "из", len(test))
# 8б. Полные дубликаты строк признаков (без ID и цели): .duplicated() помечает повторы, .sum() считает.
print("Дублей строк в train:", train.drop(columns=["customer_id", "protection_score"]).duplicated().sum())
# 8в. Пересечение ID между train и test: set() — множество, & — пересечение. Должно быть пусто!
overlap = set(train["customer_id"]) & set(test["customer_id"])
print("Общих ID между train и test:", len(overlap))
# 8г. Диапазоны ID: извлечём цифры (CUST100000 -> 100000) и сравним.
# .str.extract(r"(\d+)") находит цифры в тексте; astype(int) превращает в числа.
train_ids = train["customer_id"].str.extract(r"(\d+)", expand=False).astype(int)
test_ids = test["customer_id"].str.extract(r"(\d+)", expand=False).astype(int)
print(f"ID train: {train_ids.min()}..{train_ids.max()}, ID test: {test_ids.min()}..{test_ids.max()}")

## Выводы ноутбука 01

1. Train: 10 000 строк × 61 колонка; test: 2 000 × 60 (без ответа).
2. Признаков: ~52 числовых + 7 текстовых; цель — `protection_score` в [0, 100] без пропусков.
3. Пропуски только в `house_value`, `car_value`, `average_claim_cost` (~40–47%) — одинаково в train и test.
4. ID уникальны, дубликатов и пересечений train/test нет; ID теста идут после ID train.

Дальше — **02_eda**: графики, связи признаков с целью и проверка согласованности данных.